# YZTA Datathon — v6 Final

| | |
|---|---|
| **CV OOF (LGB)** | 1.22328 |
| **CV OOF (CAT)** | 1.22076 |
| **Weighted Avg OOF** | **1.21966** |

**Hücreleri sırayla çalıştır. Kernel restart sonrası en baştan başla.**

### v6'da Değişenler
- 4 yeni contextual feature eklendi (`uyku_baslatma_kaybi`, `kalp_stres_yuku`, `derin_uyku_orani`, `log_adim_stres`)
- CatBoost depth 6→7, learning_rate 0.01→0.015
- Ensemble: sabit ağırlık LGB=0.35 / CAT=0.65 (ters-RMSE'den daha iyi CV'de doğrulandı)

### Tasarım Kararları (Veriyle Doğrulanmış)
- **Log transform yok**: Skewness = -0.29, eşik olan 0.75'in çok altında. Önceki notebook'lardaki `USE_LOG=True` yanlıştı.
- **Winsorize yok**: Target zaten [0, 10] arasında, kırpmaya gerek yok.
- **RobustScaler yok**: LightGBM ve CatBoost ağaç tabanlıdır, ölçekleme bu modellere katkı sağlamaz.
- **Kişi bazlı normalizasyon yok**: Veri setinde her `id` eşsizdir (56000 satır, 56000 farklı kişi). Kişi başına ortalama hesaplamak için aynı kişiden birden fazla gözlem gerekir — bu yapı bu veri setinde yoktur.
- **Feature selection uygulanmadı**: %0.5 eşiğiyle `gecelik_uyanma_sayisi` ve `cinsiyet` elendiğinde CV 1.22328→1.22389'a geriledi. GB modeller önemsiz özellikleri kendi bölünme mekanizmasıyla bastırır.

## HÜCRE 1 — Kurulum & İmportlar

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm', 'catboost', '-q'],
               capture_output=True)

import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

SEED      = 42
N_FOLDS   = 5
W_LGB     = 0.35   # veriyle doğrulanmış ağırlık
W_CAT     = 0.65

np.random.seed(SEED)
print('Kütüphaneler yüklendi.')

## HÜCRE 2 — Veri Yükleme

In [ ]:
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test_x.csv')

test_id = test_raw['id'].copy()
target  = train_raw['bilissel_performans_skoru'].copy()

train = train_raw.drop(columns=['id', 'bilissel_performans_skoru']).copy()
test  = test_raw.drop(columns=['id']).copy()

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target — Min:{target.min():.2f}  Max:{target.max():.2f}  '
      f'Ort:{target.mean():.2f}  Std:{target.std():.2f}')
print(f'Skewness: {target.skew():.4f}  →  Log transform gereksiz (eşik: 0.75)')

## HÜCRE 3 — Temizlik & Encoding

In [ ]:
# İngilizce/Türkçe karışık değerleri normalize et
ulke_mapping   = {'spain':'ispanya', 'south korea':'guney kore', 'sweden':'isvec',
                  'netherlands':'hollanda', 'mexico':'meksika', 'china':'cin'}
meslek_mapping = {'lawyer': 'avukat'}

cat_cols = train.select_dtypes(include='object').columns.tolist()
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

for col in cat_cols:
    train[col] = train[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()
    test[col]  = test[col].fillna('bilinmiyor').astype(str).str.lower().str.strip()

train['ulke']   = train['ulke'].replace(ulke_mapping)
test['ulke']    = test['ulke'].replace(ulke_mapping)
train['meslek'] = train['meslek'].replace(meslek_mapping)
test['meslek']  = test['meslek'].replace(meslek_mapping)

# Eksik sayısallar → train medyanı (test sızıntısını önler)
for col in num_cols:
    med        = train[col].median()
    train[col] = train[col].fillna(med)
    test[col]  = test[col].fillna(med)

# Aykırı değer traşlama — sınırlar daima train'den
for col in num_cols:
    lo, hi     = train[col].quantile(0.01), train[col].quantile(0.99)
    train[col] = train[col].clip(lo, hi)
    test[col]  = test[col].clip(lo, hi)

# Binary encoding: sadece gerçek 2-kategorili sütunlar
for col in ['cinsiyet', 'gun_tipi']:
    cats       = sorted(pd.concat([train[col], test[col]], ignore_index=True).unique())
    mp         = {c: i for i, c in enumerate(cats)}
    train[col] = train[col].map(mp)
    test[col]  = test[col].map(mp)
    print(f'  {col}: {mp}')

# OOF Smoothed Target Encoding — nominal kategorikler için
# mevsim nominal (büyüklük ilişkisi yok) → target encoding doğru seçim
TARGET_ENCODE_COLS = ['kronotip', 'ruh_sagligi_durumu', 'meslek', 'ulke', 'mevsim']
SMOOTH             = 15
global_mean        = target.mean()
kf_enc             = KFold(n_splits=5, shuffle=True, random_state=SEED)

for col in TARGET_ENCODE_COLS:
    agg    = pd.DataFrame({'col': train[col].values, 'target': target.values})
    stats  = agg.groupby('col')['target'].agg(['mean', 'count'])
    sm_map = ((stats['count'] * stats['mean'] + SMOOTH * global_mean)
              / (stats['count'] + SMOOTH)).to_dict()

    # Train: her satır kendi foldunu görmez → sızıntı yok
    oof_enc = np.full(len(train), global_mean, dtype=np.float64)
    for tr_i, va_i in kf_enc.split(train):
        fs          = agg.iloc[tr_i].groupby('col')['target'].agg(['mean', 'count'])
        fm          = ((fs['count'] * fs['mean'] + SMOOTH * global_mean)
                       / (fs['count'] + SMOOTH)).to_dict()
        oof_enc[va_i] = train[col].iloc[va_i].map(fm).fillna(global_mean).values

    # Test: tüm train verisiyle hesaplanmış smooth ortalama
    train[col] = oof_enc
    test[col]  = test[col].map(sm_map).fillna(global_mean)
    print(f'  {col}: tamam')

print(f'\nTemizlik & encoding bitti. Train:{train.shape} | Test:{test.shape}')

## HÜCRE 4 — Feature Engineering (v3 + v4 + v6)

Toplam 21 türetilmiş özellik. Tüm v3 ve v4 korundu, 4 yeni v6 eklendi.

**v6 Yeni Özellikler:**
- `uyku_baslatma_kaybi` — Uykuya dalma gecikmesinin kaliteli uyku yüzdesine oranı (`uyku_suresi` kolonu mevcut değil, formül adapte edildi)
- `kalp_stres_yuku` — Nabız × stres: biyolojik stres yükü bütünleşik göstergesi
- `derin_uyku_orani` — Derin uykunun REM'e oranı: uyku kompozisyonu
- `log_adim_stres` — Adım × stres logaritması: büyük sayıların patlamasını önler

In [ ]:
def add_features(df):
    d = df.copy()

    # ── v3: temel etkileşimler ────────────────────────────────────────────────
    d['uyku_kalite_endeksi']  = ((d['rem_yuzdesi'] + d['derin_uyku_yuzdesi'])
                                  / (d['gecelik_uyanma_sayisi'] + 1))
    d['toplam_kaliteli_uyku'] = d['rem_yuzdesi'] + d['derin_uyku_yuzdesi']
    d['zihinsel_yuk']         = d['stres_skoru'] * d['gunluk_calisma_saati']
    d['uyku_bozulma_skoru']   = d['gecelik_uyanma_sayisi'] * d['uykuya_dalma_suresi_dk']
    d['ekran_kafein']         = d['uyku_oncesi_ekran_suresi_dk'] * d['uyku_oncesi_kafein_mg']
    d['stres_uyku_orani']     = d['stres_skoru'] / (d['uyku_kalite_endeksi'] + 1)
    d['yas_stres']            = d['yas'] * d['stres_skoru']
    d['meslek_gun_tipi']      = d['meslek'] * d['gun_tipi']

    # ── v4: aktivite ve bileşik göstergeler ──────────────────────────────────
    d['stres_aktivite_dengesi'] = d['gunluk_adim_sayisi'] / (d['stres_skoru'] + 1)
    d['uyku_kalite_kuvvet']     = d['uyku_kalite_endeksi'] * d['toplam_kaliteli_uyku']
    d['bmi_aktivite']           = d['vucut_kitle_indeksi'] / (d['gunluk_adim_sayisi'] / 1000 + 1)
    d['stres_uyku_gecikme']     = d['stres_skoru'] * d['uykuya_dalma_suresi_dk']
    d['kafein_uyanma_birikimi'] = d['uyku_oncesi_kafein_mg'] * (d['gecelik_uyanma_sayisi'] + 1)

    # ── v6: bağlamsal ve oransal özellikler ──────────────────────────────────
    # Uyku başlatma kaybı: uykuya dalma / kaliteli uyku oranı
    # (Orijinal formüldeki uyku_suresi bu veri setinde yok; adapte edildi)
    d['uyku_baslatma_kaybi'] = (d['uykuya_dalma_suresi_dk']
                                 / (d['toplam_kaliteli_uyku'] + 1))

    # Biyolojik stres yükü: nabız × stres skoru
    d['kalp_stres_yuku'] = d['dinlenik_nabiz_bpm'] * d['stres_skoru']

    # Derin uyku kalitesi: REM'e oranla derin uyku yüzdesi
    d['derin_uyku_orani'] = d['derin_uyku_yuzdesi'] / (d['rem_yuzdesi'] + 1)

    # Aktivite/stres dengesi: logaritma ile sayı patlaması engellendi
    d['log_adim_stres'] = np.log1p(d['gunluk_adim_sayisi'] * d['stres_skoru'])

    return d


train = add_features(train)
test  = add_features(test)
print(f'Feature engineering tamam. Toplam özellik: {train.shape[1]}')

## HÜCRE 5 — Feature Selection

LightGBM gain importance ile özellik katkısı hesaplanır.
**Not:** %0.5 eşiğiyle `gecelik_uyanma_sayisi` ve `cinsiyet` elindi,
ancak CV 1.22328→1.22389'a geriledi. Eşik %0.5→%0.1'e çekildi —
GB modeller zaten önemsiz özellikleri kendi bölünme mekanizmasıyla bastırır.

In [ ]:
# Hızlı selector — sadece sıfır importance'lı özellikleri at
selector = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05,
    num_leaves=31, random_state=SEED, verbose=-1
)
selector.fit(train, target)

imp_df = (pd.DataFrame({'ozellik': train.columns,
                         'importance': selector.feature_importances_})
            .sort_values('importance', ascending=False))

total_imp = imp_df['importance'].sum()
imp_df['katkı_%'] = (imp_df['importance'] / total_imp * 100).round(2)

print('Feature Importance (ilk 25):')
print(imp_df.head(25).to_string(index=False))

# Sadece sıfır katkılı özellikleri at
drop_cols = imp_df[imp_df['importance'] == 0]['ozellik'].tolist()
if drop_cols:
    print(f'\nAtılan (sıfır importance): {drop_cols}')
    train = train.drop(columns=drop_cols)
    test  = test.drop(columns=drop_cols)
else:
    print('\nHiçbir özellik atılmadı — tüm özellikler katkı sağlıyor.')

print(f'Kalan özellik sayısı: {train.shape[1]}')

## HÜCRE 6 — Model Parametreleri

In [ ]:
X    = train.copy()
Xte  = test.copy()
y    = target.copy()

# LightGBM — num_leaves:64, lr:0.015 (istendiği gibi)
lgb_params = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'verbosity'        : -1,
    'random_state'     : SEED,
    'bagging_freq'     : 5,
    'learning_rate'    : 0.015,
    'num_leaves'       : 64,
    'min_child_samples': 40,
    'feature_fraction' : 0.75,
    'bagging_fraction' : 0.75,
    'reg_alpha'        : 0.15,
    'reg_lambda'       : 0.8,
}

# CatBoost — depth:7, lr:0.015 (istendiği gibi)
cat_params = dict(
    loss_function         = 'RMSE',
    eval_metric           = 'RMSE',
    iterations            = 3000,
    learning_rate         = 0.015,
    depth                 = 7,
    l2_leaf_reg           = 6.0,
    random_strength       = 1.0,
    subsample             = 0.8,
    bootstrap_type        = 'Bernoulli',
    early_stopping_rounds = 150,
    random_seed           = SEED,
    verbose               = False,
)

print('Parametreler hazır.')
print(f'LGB: {lgb_params}')
print(f'CAT: {cat_params}')

## HÜCRE 7 — 5-Fold CV Ensemble Eğitimi

Early stopping ile 3000 iterasyon üst sınır — kaç iterasyonda durduğunu fold bazında göreceksin.

In [ ]:
N_FOLDS = 5
kf_model = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
lgb_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))
lgb_test = np.zeros(len(Xte))
cat_test = np.zeros(len(Xte))
lgb_sc, cat_sc = [], []
print('=== 5-Fold Ensemble Egitimi (LGB + CAT) ===\n')
for fold, (tr_i, va_i) in enumerate(kf_model.split(X), 1):
    Xtr, Xva = X.iloc[tr_i], X.iloc[va_i]
    ytr, yva = y.iloc[tr_i], y.iloc[va_i]
    print(f'--- Fold {fold} ---')
    # LightGBM
    dt = lgb.Dataset(Xtr, label=ytr)
    dv = lgb.Dataset(Xva, label=yva, reference=dt)
    lm = lgb.train(
        lgb_params,
        dt,
        num_boost_round=3000,
        valid_sets=[dv],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)]
    )
    lgb_oof[va_i] = lm.predict(Xva, num_iteration=lm.best_iteration)
    lgb_test += lm.predict(Xte, num_iteration=lm.best_iteration) / N_FOLDS
    lgb_fold_rmse = np.sqrt(mean_squared_error(yva, lgb_oof[va_i]))
    lgb_sc.append(lgb_fold_rmse)
    print(f'  LGB RMSE: {lgb_fold_rmse:.5f}')
    # CatBoost
    cm = CatBoostRegressor(**cat_params)
    cm.fit(Xtr, ytr, eval_set=(Xva, yva), use_best_model=True, early_stopping_rounds=150)
    cat_oof[va_i] = cm.predict(Xva)
    cat_test += cm.predict(Xte) / N_FOLDS
    cat_fold_rmse = np.sqrt(mean_squared_error(yva, cat_oof[va_i]))
    cat_sc.append(cat_fold_rmse)
    print(f'  CAT RMSE: {cat_fold_rmse:.5f}\n')
lgb_r = np.sqrt(mean_squared_error(y, lgb_oof))
cat_r = np.sqrt(mean_squared_error(y, cat_oof))
# Aliases used by HÜCRE 8 (weighted ensemble)
lgb_oof_rmse = lgb_r
cat_oof_rmse = cat_r
print('=== CV Ozeti ===')
print(f'LGB OOF RMSE: {lgb_r:.5f} (+-{np.std(lgb_sc):.5f})')
print(f'CAT OOF RMSE: {cat_r:.5f} (+-{np.std(cat_sc):.5f})')


## HÜCRE 8 — Weighted Average Ensemble

Sabit ağırlık (LGB=0.35, CAT=0.65) ile ters-RMSE ağırlığı karşılaştırılır,
düşük OOF RMSE veren kullanılır.

In [ ]:
# Sabit ağırlık — CatBoost bu veri setinde daha güçlü
wa_oof_fixed  = W_LGB * lgb_oof + W_CAT * cat_oof
wa_test_fixed = W_LGB * lgb_test + W_CAT * cat_test
rmse_fixed    = np.sqrt(mean_squared_error(y, wa_oof_fixed))

# Ters-RMSE ağırlığı — OOF skoruna göre otomatik
inv_lgb = 1.0 / (lgb_oof_rmse + 1e-12)
inv_cat = 1.0 / (cat_oof_rmse + 1e-12)
total   = inv_lgb + inv_cat
w_lgb_auto = inv_lgb / total
w_cat_auto = inv_cat / total

wa_oof_auto  = w_lgb_auto * lgb_oof + w_cat_auto * cat_oof
wa_test_auto = w_lgb_auto * lgb_test + w_cat_auto * cat_test
rmse_auto    = np.sqrt(mean_squared_error(y, wa_oof_auto))

print(f'Sabit  (LGB={W_LGB}  CAT={W_CAT})      OOF: {rmse_fixed:.5f}')
print(f'Ters-RMSE (LGB={w_lgb_auto:.3f} CAT={w_cat_auto:.3f}) OOF: {rmse_auto:.5f}')

# Daha iyi olanı seç
if rmse_fixed <= rmse_auto:
    final_test = wa_test_fixed
    final_oof  = wa_oof_fixed
    best_rmse  = rmse_fixed
    print(f'\nSeçilen: Sabit ağırlık (LGB={W_LGB} CAT={W_CAT})')
else:
    final_test = wa_test_auto
    final_oof  = wa_oof_auto
    best_rmse  = rmse_auto
    print(f'\nSeçilen: Ters-RMSE (LGB={w_lgb_auto:.3f} CAT={w_cat_auto:.3f})')

print(f'Final Ensemble OOF RMSE: {best_rmse:.5f}')

## HÜCRE 9 — Submission

In [ ]:
# Tahminleri [0, 10] aralığına sabitle
test_preds = np.clip(final_test, 0.0, 10.0)

print('Tahmin istatistikleri:')
print(f'  Min : {test_preds.min():.4f}')
print(f'  Max : {test_preds.max():.4f}')
print(f'  Ort : {test_preds.mean():.4f}')
print(f'  Std : {test_preds.std():.4f}')

submission = pd.DataFrame({
    'id'                       : test_id,
    'bilissel_performans_skoru': test_preds,
})

# Ondalık ayırıcı nokta, float_format ile 6 hane
submission.to_csv('final_master_v6.csv', index=False, float_format='%.6f')
print(f'\nfinal_master_v6.csv kaydedildi — {len(submission)} satır')
print(submission.head())